In [1]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape, Polygon
from shapely import wkt
from shapely.ops import linemerge, unary_union
alt.data_transformers.disable_max_rows()
import h3

In [2]:
with open('Data/Metro/metro_trajectories.json') as f:
    trajectories = json.load(f)


df_trajectories = []
for trajectory in trajectories['features']:
    properties = trajectory.get('properties', {})
    tram = properties.get('NOM_TRAM_LINIA')
    nom_linia = properties.get('NOM_LINIA')
    origen = properties.get('CODI_ESTACIO_INI')
    dest = properties.get('CODI_ESTACIO_FI')                   
    df_trajectories.append({
            'origen': origen,
            'dest': dest,
            'tram': tram,
            'linia': nom_linia,
            'type': 'Metro',
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories_all = pd.DataFrame(df_trajectories)
geo_df_trajectories_all = gpd.GeoDataFrame(df_trajectories_all, geometry='geometry', crs=4326)
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Inici') == False]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Final') == False]
geo_df_trajectories_all.to_crs('EPSG:25831', inplace=True)
geo_df_trajectories_all['length'] = geo_df_trajectories_all['geometry'].length 
geo_df_trajectories_all['speed'] = 25 /3.6 # 25 kmh to ms like in the paper
geo_df_trajectories_all['time'] = (geo_df_trajectories_all['length'] / geo_df_trajectories_all['speed']) /60
geo_df_trajectories_all['origen'] = 'M' + '-' + geo_df_trajectories_all['linia'] + '-' + geo_df_trajectories_all['origen'].astype(str)
geo_df_trajectories_all['dest'] = 'M' + '-' + geo_df_trajectories_all['linia'] + '-' + geo_df_trajectories_all['dest'].astype(str)
geo_df_trajectories_all['directed'] = False
geo_df_trajectories_all.to_crs('EPSG:4326', inplace=True)
geo_df_trajectories_all.to_csv("Edges/Metro.csv", index=False)